# Stage 2 — Coil optimisation

Loads `eq_fixed.h5` and `coilset.h5` and plots coil geometry, 3-D plasma+coils, and B·n on the LCFS.

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning, message=".*pynvml.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*Unequal number of field periods.*")
os.environ.setdefault("JAX_ENABLE_X64", "True")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import HTML

pio.renderers.render_on_display = False

def show3d(fig):
    return HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

from desc.equilibrium import Equilibrium
from desc.coils import CoilSet
from desc.grid import LinearGrid
from desc.objectives import QuadraticFlux
from desc.plotting import plot_surfaces, plot_3d, plot_coils

HERE = Path(".")

In [ ]:
eq_fixed = Equilibrium.load(str(HERE / "eq_fixed.h5"))
coilset  = CoilSet.load(str(HERE / "coilset.h5"))
print(f"Equilibrium: NFP={eq_fixed.NFP}, L={eq_fixed.L}, M={eq_fixed.M}, N={eq_fixed.N}")
print(f"Coilset    : {len(coilset.coils)} coils")

## Coil geometry

In [ ]:
lengths  = np.array([c.length  for c in coilset.coils])
currents = np.array([c.current for c in coilset.coils])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(len(lengths)),  lengths,  color="steelblue")
axes[0].set_xlabel("coil index"); axes[0].set_ylabel("length [m]")
axes[0].set_title("Coil lengths")
axes[1].bar(range(len(currents)), currents, color="tomato")
axes[1].set_xlabel("coil index"); axes[1].set_ylabel("current [A]")
axes[1].set_title("Coil currents")
fig.suptitle("Stage 2 — optimised coil properties", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
N_phi = eq_fixed.NFP * 2
fig, axes = plot_surfaces(
    eq_fixed,
    phi=N_phi,
    rho=np.array([1.0]),
    theta=0,
)
fig.suptitle("Stage 2 — LCFS at coil toroidal angles", fontsize=13)
plt.tight_layout()
plt.show()

## 3-D — coils + plasma surface

In [ ]:
fig = plot_3d(eq_fixed, "|B|", alpha=0.4)
plot_coils(coilset, fig=fig)
fig.update_layout(title="Stage 2 — plasma surface + optimised coils")
show3d(fig)

## B·n map on the LCFS

In [ ]:
fig = plot_3d(
    eq_fixed,
    "B*n",
    field=coilset,
    field_grid=LinearGrid(N=50, NFP=eq_fixed.NFP),
)
plot_coils(coilset, fig=fig)
fig.update_layout(title="Stage 2 — B·n on LCFS after coil optimisation [T]")
show3d(fig)

In [ ]:
obj = QuadraticFlux(eq=eq_fixed, field=coilset, vacuum=True)
obj.build(verbose=0)
Bn_rms = float(np.sqrt(obj.compute_scalar(*obj.xs(eq_fixed, coilset))))
print(f"B·n RMS on LCFS : {Bn_rms:.4e} T")